# Remote NVIDIA GPU smoke test

In this notebook, I want to run a Python function on a remote NVIDIA L4 GPU from Mac.

# Key Takeaways

- The GPU belongs to the physical host, not the container.
- The container receives access to GPU device files and driver libraries.
- The CUDA image contains user-space CUDA software.
- The NVIDIA kernel driver runs on the host.
- `nvidia-smi` is a normal executable that communicates with NVML and the driver.
- Defining a Modal function does not allocate a GPU.
- The GPU is requested when `.remote()` executes.
- GPU operations are generally asynchronous.
- Operations such as `.item()` can force CPU–GPU synchronization.

# Mental Model

Mac
↓
Modal API
↓
Remote Linux container
↓
Python process
↓
CUDA user-space libraries
↓
NVIDIA host driver
↓
Physical GPU

In [1]:
%pip install -q "modal>=1.5,<2"


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
!modal setup


     ▟█▀▀▀▀▜▖  ▗█▛▀▀▀▀▙
    ▟▘▝▙    ▜▖▗▛ ▜▖   ▝▙
   ▟▘  ▝▙▄▄▄▄█▛   ▜▖   ▝▙
  ▟▘   ▟▘   ▗▛▜▖   ▜▖   ▝▙
 ▟▘   ▟▘   ▗▛  ▜▖   ▜▖   ▝▙
▟▘   ▟▘   ▗▛    ▜▖   ▜▄▄▄▄▟▙
▝▙  ▟▘   ▗▛      ▜▖ ▗▛    ▟▘
 ▝▙▟▘   ▗▛        ▜▄▛    ▟▘
  ▝▀▀▀▀▀▀          ▀▀▀▀▀▀▘

The web browser should have opened for you to authenticate and get an API token.
If it didn't, please copy this URL into your web browser manually:

]8;id=442994;https://modal.com/token-flow/tf-UuwzK7f4Kd66AfWWjnF4qT\https://modal.com/token-flow/tf-UuwzK7f4Kd66AfWWjnF4qT]8;;\

⠋ Waiting for authentication in the web browser
⠋ Waiting for token flow to complete...
Web authentication finished successfully!
Token is connected to the vishalkt07 workspace.
Verifying token against https://api.modal.com,https://api.modal2.com
Token verified successfully!
⠋ Storing token
Token written to /Users/vishal/.modal.toml in profile vishalkt07.


In [5]:
# Inspect the local environment

import sys
import modal

LOCAL_PYTHON = f"{sys.version_info.major}.{sys.version_info.minor}"

print("Local Python:", LOCAL_PYTHON)
print("Modal version:", modal.__version__)

Local Python: 3.12
Modal version: 1.5.3


In [6]:
# Create the Modal application

app = modal.App("gpu-inference-systems")

In [7]:
# Describe the remote container

CUDA_IMAGE = "nvidia/cuda:12.8.1-base-ubuntu22.04"

image = (
    modal.Image.from_registry(
        CUDA_IMAGE,
        add_python=LOCAL_PYTHON,
    )
    .entrypoint([])
)

print("Remote image:", CUDA_IMAGE)
print("Remote Python:", LOCAL_PYTHON)

Remote image: nvidia/cuda:12.8.1-base-ubuntu22.04
Remote Python: 3.12


In [9]:
# Define the remote function

@app.function(
    image=image,
    gpu="L4",
    timeout=300
)
def inspect_gpu() -> dict[str, object]:
    import subprocess

    completed = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total,compute_cap",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    raw_output = completed.stdout.strip()
    fields = [field.strip() for field in raw_output.split(",")]
    if len(fields) != 4:
        raise RuntimeError(
            f"Expected four fields, received: {raw_output!r}"
        )
    return {
        "gpu_name": fields[0],
        "driver_version": fields[1],
        "memory_mib": int(fields[2]),
        "compute_capability": fields[3],
        "gpu_visible": True,
    }

In [10]:
# Execute remotely

with modal.enable_output():
    with app.run():
        gpu_info = inspect_gpu.remote()

✓ Initialized. View run at 
https://modal.com/apps/vishalkt07/main/ap-qz5QpjJVLEyOaryHhboLf0
⠋ Initializing...
Building image im-SlDIENrtbQTYPygCnDzP4x
⠙ Creating objects....
=> Step 0: FROM nvidia/cuda:12.8.1-base-ubuntu22.04
Getting image source signatures
Copying blob sha256:73389fbd088f5ed5d9fd258baced59de092978b4f483920ea6d074522a105119
Copying blob sha256:b95112eaf2836b9c7193858419f6d17933a873c4a2c9227dc497f5e090ea9c87
Copying blob sha256:030ef82509363afebdc78482e6f8c07f2bc8b65a45caf082c2a9a621c7deae63
Copying blob sha256:72ac9ccfda38d144ef8e3aadbd5851e7fa9aacd261f8e15d7365db0f816d3433
Copying blob sha256:9cb31e2e37eab1bff50f727e979fcacb509e225fb853433a6fe21d2fb34e6305
Copying config sha256:c6b34222dc09b46d139774ca08e4ccc0ced8810948f2d343a6ece49c8efdd89e
Writing manifest to image destination
Unpacking OCI image...
   • unpacking rootfs ...     
   • ... done                 
   • unpacked image rootfs: /tmp/.tmpgCqeSf
Copied image in 731.83ms0m
m⠧ Creating objects...
=> Step 1: C

In [11]:
gpu_info

{'gpu_name': 'NVIDIA L4',
 'driver_version': '580.95.05',
 'memory_mib': 23034,
 'compute_capability': '8.9',
 'gpu_visible': True}

In [12]:
assert gpu_info["gpu_visible"] is True
assert "L4" in gpu_info["gpu_name"]
assert gpu_info["memory_mib"] > 0
assert gpu_info["driver_version"]
assert gpu_info["compute_capability"]

print("PASS: remote NVIDIA L4 verified")

PASS: remote NVIDIA L4 verified


In [13]:
from pathlib import Path
import json

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

artifact_path = ARTIFACTS / "gpu_environment.json"

artifact_path.write_text(
    json.dumps(gpu_info, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("Saved:", artifact_path.resolve())

Saved: /Users/vishal/workspace/gpu-inference-systems/remote-gpu-smoke-test/artifacts/gpu_environment.json


In [14]:
print(artifact_path.read_text(encoding="utf-8"))

{
  "compute_capability": "8.9",
  "driver_version": "580.95.05",
  "gpu_name": "NVIDIA L4",
  "gpu_visible": true,
  "memory_mib": 23034
}
